# 01 · Data Cleaning: U.S. Coal Consumption & Quality (EIA, 2020–2025)

**Goal:** turn the raw EIA file into clean, clearly labeled tables we can analyze without double-counting or unit mistakes.

**Input:** `data/raw/eia_coal_consumption_quality_2020_2025.csv` (1,750 rows, untouched)

**Outputs (in `data/clean/`):**

| File | What it is | Safe to add up? |
|---|---|---|
| `coal_clean.csv` | Every row, cleaned, with labels for geography level and sector level | Only after filtering to one level |
| `us_by_sector.csv` | U.S. totals for the 4 top-level sectors, by year | ✅ Yes, across sectors |
| `states_by_sector.csv` | 48 states + DC (no MA or VT rows), 4 top-level sectors, by year | ✅ Yes, across states *or* across sectors |
| `power_subsectors_us.csv` | U.S. power sector split: utilities vs. independent power producers | ✅ Yes (Utility + IPP = Electric Power) |

The three shortcut files are exact filters of `coal_clean.csv`, minus the label columns that would be the same on every row.

Every cleaning decision below has a short explanation so the team (and the judges) can see *why*.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

RAW = "../data/raw/eia_coal_consumption_quality_2020_2025.csv"
OUT = "../data/clean/"

raw = pd.read_csv(RAW)
print(raw.shape)
raw.head()

(1750, 19)


,period,location,stateDescription,sector,sectorDescription,ash-content,consumption,heat-content,price,receipts,stocks,sulfur-content,ash-content-units,consumption-units,heat-content-units,price-units,receipts-units,stocks-units,sulfur-content-units
0,2020,90,Pacific,1,Electric Utility,6.30,1541461.0,16.2558,42.25,873007.0,NaN,0.21,average ash percent by weight,short tons,average sulfur percent by weight,dollars per short ton,short tons,short tons,average sulfur percent by weight
1,2020,90,Pacific,2,IPP Non-CHP,8.36,3476488.0,16.6534,NaN,3525648.0,NaN,0.40,average ash percent by weight,short tons,average sulfur percent by weight,dollars per short ton,short tons,short tons,average sulfur percent by weight
2,2020,90,Pacific,3,IPP CHP,4.92,890674.0,18.6360,NaN,725275.0,NaN,0.27,average ash percent by weight,short tons,average sulfur percent by weight,dollars per short ton,short tons,short tons,average sulfur percent by weight
3,2020,90,Pacific,94,Independent Power Producers,7.70,4367162.0,17.0577,NaN,4250923.0,NaN,0.38,average ash percent by weight,short tons,average sulfur percent by weight,dollars per short ton,short tons,short tons,average sulfur percent by weight
4,2020,90,Pacific,98,Electric Power,7.48,5908623.0,16.8485,NaN,5123930.0,NaN,0.35,average ash percent by weight,short tons,average sulfur percent by weight,dollars per short ton,short tons,short tons,average sulfur percent by weight


## Step 1: Rename columns

The raw names mix hyphens and camelCase. We rename everything to `snake_case` and put the unit in the name, so nobody has to check a separate units column.

In [2]:
df = raw.rename(columns={
    "period": "year",
    "location": "location_code",
    "stateDescription": "location_name",
    "sector": "sector_code",
    "sectorDescription": "sector_name",
    "consumption": "consumption_tons",
    "receipts": "receipts_tons",
    "stocks": "stocks_tons",
    "price": "price_usd_per_ton",
    "heat-content": "heat_raw",          # fixed in Step 4
    "sulfur-content": "sulfur_pct",
    "ash-content": "ash_pct",
})
df["location_code"] = df["location_code"].astype(str)
df.columns.tolist()

['year',
 'location_code',
 'location_name',
 'sector_code',
 'sector_name',
 'ash_pct',
 'consumption_tons',
 'heat_raw',
 'price_usd_per_ton',
 'receipts_tons',
 'stocks_tons',
 'sulfur_pct',
 'ash-content-units',
 'consumption-units',
 'heat-content-units',
 'price-units',
 'receipts-units',
 'stocks-units',
 'sulfur-content-units']

## Step 2: Check the units

The data guide says volumes are in *thousands* of short tons. The units columns disagree, so we check what's actually in them.

In [3]:
unit_cols = [c for c in df.columns if c.endswith("-units")]
for c in unit_cols:
    print(f"{c:25s} -> {df[c].unique().tolist()}")

ash-content-units         -> ['average ash percent by weight']
consumption-units         -> ['short tons']
heat-content-units        -> ['average sulfur percent by weight']
price-units               -> ['dollars per short ton']
receipts-units            -> ['short tons']
stocks-units              -> ['short tons']
sulfur-content-units      -> ['average sulfur percent by weight']


In [4]:
us_power_2020 = df.query("location_code == 'US' and sector_name == 'Electric Power' and year == 2020")
print(f"U.S. power-sector coal burned in 2020: {us_power_2020['consumption_tons'].iloc[0]:,.0f}")

U.S. power-sector coal burned in 2020: 435,826,849


**Findings:**
- Volumes are in **plain short tons**, not thousands. The U.S. power sector burned ~436 *million* tons in 2020, which matches the known scale of U.S. coal burn. If the values were thousands of tons, that would be 436 *billion* tons, which is impossible. **We go by the units column, not the data guide.**
- `heat-content-units` is mislabeled as *"average sulfur percent by weight"*, which is clearly a copy-paste error. Step 4 sorts out the real heat units.

All units are now in the column names, so we drop the units columns.

In [5]:
df = df.drop(columns=unit_cols)

## Step 3: Label geography and sector levels (prevents double-counting)

The file stacks several levels on top of each other:

**Geography:** U.S. Total → Census regions → states. If you add all rows together, every ton of coal gets counted about 3 times.

**Sectors:** `Electric Power` is already the sum of `Electric Utility` + `Independent Power Producers`, and IPP is itself `IPP Non-CHP` + `IPP CHP`.

We add label columns so every analysis can filter to **one level** first.

In [6]:
DIVISIONS = {"NEW", "MAT", "ENC", "WNC", "SAT", "ESC", "WSC", "MTN", "PCC", "PCN"}

def geo_level(code):
    if code == "US":
        return "national"
    if code in DIVISIONS:
        return "region"            # 10 non-overlapping regions that add up to U.S.
    if code == "90":
        return "region_pacific_total"  # = PCC + PCN; overlaps them, so it's kept separate
    if code == "PR":
        return "territory"         # Puerto Rico: NOT included in the U.S. Total row
    return "state"                 # 48 states + DC (MA and VT have no rows)

df["geo_level"] = df["location_code"].map(geo_level)
df.loc[df["location_code"] == "90", "location_code"] = "PAC"

SECTORS = {
    # code: (level, parent, friendly group)
    98: ("top", None, "Power"),
    10: ("top", None, "Industrial"),
    9:  ("top", None, "Coke (steel)"),
    8:  ("top", None, "Commercial"),
    1:  ("sub", "Electric Power", "Power"),
    94: ("sub", "Electric Power", "Power"),
    2:  ("sub_sub", "Independent Power Producers", "Power"),
    3:  ("sub_sub", "Independent Power Producers", "Power"),
}
df["sector_level"]  = df["sector_code"].map(lambda c: SECTORS[c][0])
df["sector_parent"] = df["sector_code"].map(lambda c: SECTORS[c][1])
df["sector_group"]  = df["sector_code"].map(lambda c: SECTORS[c][2])

print(df.groupby("geo_level")["location_code"].nunique())
df[["sector_code", "sector_name", "sector_level", "sector_parent", "sector_group"]].drop_duplicates().sort_values("sector_code")

geo_level
national                 1
region                  10
region_pacific_total     1
state                   49
territory                1
Name: location_code, dtype: int64


,sector_code,sector_name,sector_level,sector_parent,sector_group
0,1,Electric Utility,sub,Electric Power,Power
1,2,IPP Non-CHP,sub_sub,Independent Power Producers,Power
2,3,IPP CHP,sub_sub,Independent Power Producers,Power
8,8,Commercial and Institutional,top,NaN,Commercial
13,9,Coke Plants,top,NaN,Coke (steel)
6,10,Other Industrial,top,NaN,Industrial
3,94,Independent Power Producers,sub,Electric Power,Power
4,98,Electric Power,top,NaN,Power


## Step 4: Fix heat content (two units mixed in one column)

Heat-content values fall into two groups that are far apart: about 9–25, and about 5,000–15,000.

In [7]:
print(pd.cut(df["heat_raw"], [-1, 0, 30, 1000, 20000]).value_counts().sort_index())
df.groupby("sector_name")["heat_raw"].median().round(1)

heat_raw
(-1, 0]           177
(0, 30]          1051
(30, 1000]          0
(1000, 20000]     495
Name: count, dtype: int64


sector_name
Coke Plants                     14431.0
Commercial and Institutional    11234.0
Electric Power                     18.2
Electric Utility                   18.0
IPP CHP                            15.1
IPP Non-CHP                        17.2
Independent Power Producers        17.2
Other Industrial                11570.0
Name: heat_raw, dtype: float64

The split follows sector exactly:
- **Power sectors** report **million Btu per short ton** (~18 MMBtu/ton)
- **Industrial, coke, and commercial** report **Btu per pound** (~11,000–14,500 Btu/lb)

The conversion is: 1 short ton = 2,000 lb, so **MMBtu/ton × 1,000,000 ÷ 2,000 = MMBtu/ton × 500 = Btu/lb**.

We put everything in Btu/lb (the unit the data guide describes) and also keep an MMBtu/ton version, which we need for price-per-energy later. Zeros are handled in Step 5.

In [8]:
small_scale = df["heat_raw"].between(0.001, 100)
power_codes = {98, 1, 94, 2, 3}

# safety check: the small-scale values should be exactly the power sectors
assert set(df.loc[small_scale, "sector_code"]) <= power_codes
assert not (df.loc[df["heat_raw"] > 100, "sector_code"].isin(power_codes)).any()

df["heat_btu_per_lb"] = np.where(small_scale, df["heat_raw"] * 500, df["heat_raw"]).round(2)
df["heat_mmbtu_per_ton"] = (df["heat_btu_per_lb"] / 500).round(4)
df["heat_was_converted"] = small_scale
df = df.drop(columns="heat_raw")

df.groupby("sector_name")["heat_btu_per_lb"].median().round(0)

sector_name
Coke Plants                     14431.0
Commercial and Institutional    11234.0
Electric Power                   9083.0
Electric Utility                 9024.0
IPP CHP                          7557.0
IPP Non-CHP                      8596.0
Independent Power Producers      8605.0
Other Industrial                11570.0
Name: heat_btu_per_lb, dtype: float64

After conversion the numbers make sense: power plants burn mostly lower-energy coal (~9,000–9,400 Btu/lb, largely Western sub-bituminous), and coke plants buy high-energy metallurgical coal (~14,400 Btu/lb).

## Step 5: Turn impossible zeros into "missing"

The guide says blanks mean *unknown*, not zero. But some cells hold **0 where 0 can't be true**:

| Column | Why 0 isn't real | Rows |
|---|---|---|
| `heat_btu_per_lb` | All coal holds energy. These rows burned no coal, so there's no coal to measure. | see below |
| `sulfur_pct`, `ash_pct` | Same reason: no coal was delivered or burned to measure. | |
| `price_usd_per_ton` | Coal isn't free. Most of these rows had 0 deliveries, so there's no price to report. | |

We set these to blank (NaN) so they don't drag down averages.

**We keep 0 for consumption and receipts.** "This state burned no coal in this sector" is a real and important fact (for example, California's power sector).

In [9]:
zero_counts = {}
for col in ["heat_btu_per_lb", "heat_mmbtu_per_ton", "sulfur_pct", "ash_pct", "price_usd_per_ton"]:
    mask = df[col] == 0
    zero_counts[col] = int(mask.sum())
    df.loc[mask, col] = np.nan
zero_counts

{'heat_btu_per_lb': 177,
 'heat_mmbtu_per_ton': 177,
 'sulfur_pct': 63,
 'ash_pct': 63,
 'price_usd_per_ton': 17}

## Step 6: Look at what's still missing (don't fill it in)

We **don't** invent values for blanks, because guessing would put made-up numbers in the analysis. Instead we measure where the gaps are so we know what we can and can't say.

In [10]:
coverage = (df.notna()
              .groupby(df["sector_name"])[["consumption_tons", "receipts_tons", "stocks_tons", "price_usd_per_ton", "heat_btu_per_lb"]]
              .mean().mul(100).round(0))
coverage.columns = [c.split("_")[0] + " %" for c in coverage.columns]
coverage

,consumption %,receipts %,stocks %,price %,heat %
sector_name,,,,,
Coke Plants,100.0,100.0,100.0,19.0,100.0
Commercial and Institutional,90.0,90.0,98.0,23.0,90.0
Electric Power,100.0,93.0,2.0,46.0,93.0
Electric Utility,100.0,93.0,2.0,76.0,90.0
IPP CHP,100.0,67.0,4.0,4.0,65.0
IPP Non-CHP,100.0,85.0,3.0,2.0,84.0
Independent Power Producers,100.0,81.0,2.0,18.0,83.0
Other Industrial,100.0,100.0,95.0,55.0,96.0


In [11]:
price_rows = (df.assign(has_price=df["price_usd_per_ton"].notna())
                .pivot_table(index="sector_name", columns="year", values="has_price", aggfunc="sum"))
print("Number of rows WITH a price, by sector and year:")
price_rows

Number of rows WITH a price, by sector and year:


year,2020,2021,2022,2023,2024,2025
sector_name,,,,,,
Coke Plants,3,3,2,2,2,2
Commercial and Institutional,7,6,7,4,4,4
Electric Power,30,0,30,33,36,34
Electric Utility,49,0,42,42,42,42
IPP CHP,1,0,1,1,1,1
IPP Non-CHP,1,0,1,1,1,1
Independent Power Producers,7,0,8,10,11,10
Other Industrial,31,30,30,29,27,21


In [12]:
stock_geo = df[df["stocks_tons"].notna()].groupby("sector_name")["geo_level"].unique()
print("Where stockpile data exists:")
stock_geo

Where stockpile data exists:


sector_name
Coke Plants                     [state, region, national]
Commercial and Institutional    [state, region, national]
Electric Power                                 [national]
Electric Utility                               [national]
IPP CHP                                        [national]
IPP Non-CHP                                    [national]
Independent Power Producers                    [national]
Other Industrial                [state, region, national]
Name: geo_level, dtype: object

**Key gaps to put on the *Data Limitations* slide:**
1. **No power-sector prices in 2021.** Every power row is blank that year, so price charts for power will skip 2021 (shown as a gap, not a zero).
2. **Power-sector stockpiles exist only at the national level**, so stockpile analysis by state is limited to industrial and coke users.
3. **Price is blank in about 60% of rows**, mostly small states where EIA withholds data to protect the privacy of individual plants.
4. **Coke plants report data in only a few states** (AL, IL, IN, MI, OH, PA, VA, WV).
5. **2025 is preliminary** and may be revised.
6. **States leave the data when their coal use ends.** For example, Hawaii disappears after 2022 when its last coal plant closed. A missing row there means "stopped," not "unknown."

## Step 7: Add helper columns for the analysis

In [13]:
df["is_preliminary"] = df["year"] == 2025

# Are deliveries keeping up with burn? Positive = stockpiling, negative = drawing down piles
df["receipts_minus_consumption_tons"] = (df["receipts_tons"] - df["consumption_tons"]).round(2)

# Days of burn: how many days the year-end stockpile would last at this year's burn rate
burn_per_day = df["consumption_tons"] / 365
df["days_of_burn"] = pd.Series(np.where(burn_per_day > 0, df["stocks_tons"] / burn_per_day, np.nan)).round(1)

# Price per unit of energy: lets us compare cheap low-energy coal with pricey high-energy coal fairly
df["price_usd_per_mmbtu"] = (df["price_usd_per_ton"] / df["heat_mmbtu_per_ton"]).round(3)

# Millions of tons, easier to read on slides
for c in ["consumption", "receipts", "stocks"]:
    df[f"{c}_mm_tons"] = (df[f"{c}_tons"] / 1e6).round(3)

df["no_coal_burned"] = df["consumption_tons"] == 0

We round only the columns **we calculated** (to a sensible number of decimals), so the files are easy to read. The original EIA columns (tons, price, sulfur, ash) are left exactly as published.

## Step 8: Validation checks

These checks confirm the labels from Step 3 are right: within each sector and year, the parts should add up to the whole.

In [14]:
def compare(level_mask, label):
    parts = df[level_mask].groupby(["year", "sector_name"])[["consumption_tons", "receipts_tons"]].sum()
    total = df[df["geo_level"] == "national"].set_index(["year", "sector_name"])[["consumption_tons", "receipts_tons"]]
    pct_diff = ((parts - total).abs() / total).max() * 100
    print(f"{label:35s} max difference vs U.S. Total: consumption {pct_diff['consumption_tons']:.3f}%, receipts {pct_diff['receipts_tons']:.3f}%")

compare(df["geo_level"] == "state", "48 states + DC (no MA or VT rows)")
compare(df["geo_level"] == "region", "10 regions")
compare(df["geo_level"].isin(["state", "territory"]), "states + Puerto Rico (should be off)")

48 states + DC (no MA or VT rows)   max difference vs U.S. Total: consumption 0.000%, receipts 0.000%
10 regions                          max difference vs U.S. Total: consumption 0.000%, receipts 0.000%
states + Puerto Rico (should be off) max difference vs U.S. Total: consumption 1.766%, receipts 1.691%


In [15]:
p = df.pivot_table(index=["year", "location_code"], columns="sector_name", values="consumption_tons")
ep_gap  = (p["Electric Utility"].fillna(0) + p["Independent Power Producers"].fillna(0) - p["Electric Power"]).abs().max()
ipp_gap = (p["IPP Non-CHP"].fillna(0) + p["IPP CHP"].fillna(0) - p["Independent Power Producers"]).abs().max()
print(f"Utility + IPP vs Electric Power, worst gap: {ep_gap:,.0f} tons")
print(f"IPP Non-CHP + IPP CHP vs IPP, worst gap:   {ipp_gap:,.0f} tons")
assert ep_gap < 1 and ipp_gap < 1
assert not df.duplicated(["year", "location_code", "sector_code"]).any(), "duplicate rows!"
print("All checks passed ✅")

Utility + IPP vs Electric Power, worst gap: 0 tons
IPP Non-CHP + IPP CHP vs IPP, worst gap:   0 tons
All checks passed ✅


**Result:** states and regions both add up to the U.S. Total, and the power sub-sectors add up to Electric Power. The U.S. Total **does not include Puerto Rico** (~1.5 M tons/yr), so adding PR to the states makes the sum too high. That's why PR has its own `territory` label.

## Step 9: Save the clean files

In [16]:
col_order = [
    "year", "is_preliminary", "location_code", "location_name", "geo_level",
    "sector_code", "sector_name", "sector_level", "sector_parent", "sector_group",
    "consumption_tons", "receipts_tons", "stocks_tons",
    "consumption_mm_tons", "receipts_mm_tons", "stocks_mm_tons",
    "receipts_minus_consumption_tons", "days_of_burn", "no_coal_burned",
    "price_usd_per_ton", "price_usd_per_mmbtu",
    "heat_btu_per_lb", "heat_mmbtu_per_ton", "heat_was_converted",
    "sulfur_pct", "ash_pct",
]
geo_order = {"national": 0, "region": 1, "region_pacific_total": 2, "state": 3, "territory": 4}
clean = (df[col_order]
         .assign(_g=df["geo_level"].map(geo_order))
         .sort_values(["year", "_g", "location_code", "sector_code"])
         .drop(columns="_g")
         .reset_index(drop=True))

# The shortcut files are already filtered to one level, so the label columns would just repeat
# the same value on every row. We drop them to keep these files simple.
label_cols = ["geo_level", "sector_code", "sector_level", "sector_parent", "heat_was_converted"]
us_by_sector = clean.query("geo_level == 'national' and sector_level == 'top'").drop(columns=label_cols)
states_by_sector = clean.query("geo_level == 'state' and sector_level == 'top'").drop(columns=label_cols)
power_subsectors_us = clean.query("geo_level == 'national' and sector_level == 'sub'").drop(columns=label_cols)

clean.to_csv(OUT + "coal_clean.csv", index=False)
us_by_sector.to_csv(OUT + "us_by_sector.csv", index=False)
states_by_sector.to_csv(OUT + "states_by_sector.csv", index=False)
power_subsectors_us.to_csv(OUT + "power_subsectors_us.csv", index=False)

for name, t in [("coal_clean", clean), ("us_by_sector", us_by_sector),
                ("states_by_sector", states_by_sector), ("power_subsectors_us", power_subsectors_us)]:
    print(f"{name:22s} {t.shape[0]:5d} rows x {t.shape[1]} cols")

coal_clean              1750 rows x 26 cols
us_by_sector              24 rows x 21 cols
states_by_sector         657 rows x 21 cols
power_subsectors_us       12 rows x 21 cols


## Sanity check: first look at the clean national data

In [17]:
(us_by_sector.pivot_table(index="sector_name", columns="year", values="consumption_mm_tons")
             .round(1).sort_values(2020, ascending=False))

year,2020,2021,2022,2023,2024,2025
sector_name,,,,,,
Electric Power,435.8,501.4,472.8,387.2,373.3,415.4
Other Industrial,25.7,25.8,25.9,22.2,21.5,20.1
Coke Plants,14.4,17.6,16.0,15.8,15.5,14.6
Commercial and Institutional,0.8,0.8,0.8,0.7,0.6,0.7


In [18]:
(us_by_sector.pivot_table(index="sector_name", columns="year", values="price_usd_per_ton")
             .round(2).sort_values(2020, ascending=False))

year,2020,2021,2022,2023,2024,2025
sector_name,,,,,,
Coke Plants,127.16,122.37,195.61,211.60,189.68,183.43
Commercial and Institutional,84.52,83.00,89.85,95.76,99.20,102.06
Other Industrial,57.18,57.89,69.76,83.39,78.04,76.43
Electric Power,36.13,NaN,44.50,46.96,46.45,45.56


Next notebook: `02_analysis.ipynb` covers consumption trends, deliveries vs. burn, stockpiles, prices, and quality.